<a href="https://colab.research.google.com/github/wangdian412/-/blob/master/clang_m3_output_parsers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Make sure that you have set up API KEYS in Secrets on the left menu icon.
from google.colab import userdata
openai_api_key = userdata.get('OPENAI_API_KEY')

In [ ]:
!pip install -q python-dotenv langchain openai langchain_openai

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, CommaSeparatedListOutputParser, JsonOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field

In [ ]:
model = ChatOpenAI(model="gpt-3.5-turbo-1106",
                   temperature=0.7,
                   openai_api_key=os.getenv('OPENAI_API_TOKEN'))

In [ ]:
def call_string_output_parser():
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Tell me a joke about the following subject"),
        ("human", "{input}")
    ])
    parser = StrOutputParser()
    chain = prompt | model | parser
    return chain.invoke({
        "input": "dog"
    })

In [ ]:
def call_list_output_parser():
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Generate a list of 10 synonyms for the following word. Return the results as a comma seperated list."),
        ("human", "{input}")
    ])
    parser = CommaSeparatedListOutputParser()

    chain = prompt | model | parser
    return chain.invoke({
        "input": "happy"
    })

In [ ]:
def call_json_output_parser():
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Extract information from the following phrase.\nFormatting Instructions: {format_instructions}"),
        ("human", "{phrase}")
    ])
    class Person(BaseModel):
        recipe: str = Field(description="the name of the recipe")
        ingredients: list = Field(description="ingredients")

    parser = JsonOutputParser(pydantic_object=Person)
    chain = prompt | model | parser

    return chain.invoke({
        "phrase": "The ingredients for a Margherita pizza are tomatoes, onions, cheese, basil",
        "format_instructions": parser.get_format_instructions()
    })

print(type(call_string_output_parser()))<br>
print(type(call_list_output_parser()))

In [ ]:
print(call_json_output_parser())